In [0]:
from pyspark.sql import functions as F

LANDING = "/Volumes/workspace/nsw_property/landing/sales/"

raw = (
    spark.read.text(LANDING + "*.DAT")
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

records = (
    raw
    .withColumn("fields", F.split("value", ";"))
    .withColumn("record_type", F.col("fields")[0])
)


In [0]:
def field(i):
    """i deh talbariig avna (0 ees ehlej toolno). Hooson bol null bolgono."""
    value = F.trim(F.col("fields")[i])
    return F.when(value == "", None).otherwise(value)

In [0]:
sales = (
    records
    .where(F.col("record_type") =="B")
    .select(
        field(1).alias("district_code"),
        field(2).cast("bigint").alias("property_id"),
        field(3).cast("int").alias("sales_counter"),
        field(6).alias("unit_number"),
        field(7).alias("house_number"),
        field(8).alias("street_name"),
        field(9).alias("suburb"),
        field(10).alias("post_code"),
        field(11).cast("double").alias("area"),
        field(12).alias("area_type"),
        F.to_date(field(13), "yyyyMMdd").alias("contract_date"),
        F.to_date(field(14), "yyyyMMdd").alias("settlement_date"),
        field(15).cast("bigint").alias("purchase_price"),
        field(17).alias("nature_of_property"),
        field(23).alias("dealing_number"),
        "_source_file"
    )
)

display(sales)

In [0]:
sales = (
    sales
    .withColumn(
        "area_sqm",
        F.when(F.col("area_type") == "H", F.col("area") * 10000)
         .otherwise(F.col("area")),
    )
    .withColumn(
        "address",
        F.concat_ws(" ", "unit_number", "house_number", "street_name"),
    )
)

display(
    sales.select(
        "address", "suburb", "purchase_price",
        "contract_date", "area_sqm", "nature_of_property",
    )
)

In [0]:
display(sales.orderBy(F.col("purchase_price").desc()).limit(1))

In [0]:
display(sales.orderBy(F.col("contract_date").asc()).limit(1))

In [0]:
legal = (
    records
    .where(F.col("record_type") == "C")
    .select(
        field(1).alias("district_code"),
        field(2).alias("property_id"),
        field(3).cast("int").alias("sales_counter"),
        field(5).alias("legal_description"),
        "_source_file"
    )
)

display(legal)

In [0]:
parties = (
    records
    .where(F.col("record_type") == "D")
    .select(
        field(1).alias("district_code"),
        field(2).cast("bigint").alias("property_id"),
        field(3).cast("int").alias("sales_counter"),
        field(5).alias("party_type"),
        "_source_file"
    )
    .groupBy("district_code", "property_id", "sales_counter", "_source_file")
    .agg(
        F.count_if(F.col("party_type") == "P").alias("purchaser_count"),
        F.count_if(F.col("party_type") == "V").alias("vendor_count"),
    )
)

display(parties)

In [0]:
SALE_KEY = ["district_code", "property_id", "sales_counter", "_source_file"]

display(legal.groupBy(SALE_KEY).count().where("count>1"))

In [0]:
sales_full = (
    sales
    .join(legal, SALE_KEY, "left")
    .join(parties, SALE_KEY, "left")
)

print("Muriin too: ", sales_full.count())

display(
    sales_full.select(
        "address", "suburb", "purchase_price",
        "legal_description", "purchaser_count", "vendor_count"
    )
)